<a href="https://colab.research.google.com/github/LuciaKajanova/dspracticum25_flowers_team/blob/maritn-upravy/gemma3_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning GEMMY3 na datech

In [1]:
# ==============================================================================
# 0. KONFIGURACE A INSTALACE
# ==============================================================================
# --- Konfigurace souborů (Musí být nahrány do Colab adresáře) ---
TRAIN_FILE = "train_100.jsonl"
TEST_FILE = "test_99.jsonl"
OUTPUT_DIR = "gemma_3_4b_pisne_lora" # Adresář pro uložení modelu

# --- Instalace Unsloth a potřebných knihoven ---
print("1. Instaluji knihovny...")
# Používáme stabilní a rychlou instalaci pro Colab
!pip install -q --upgrade "unsloth[colab-new]>=2024.7" torch trl datasets accelerate peft bitsandbytes

import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import json
import os
import random
print("Importy hotovy. Pokračuji načtením modelu.")

# --- Globální konfigurace modelu ---
max_seq_length = 2048
dtype = None # None pro auto detekci (bfloat16 na A100/H100, float16 na T4/V100)
load_in_4bit = True # QLoRA pro úsporu VRAM


# ==============================================================================
# 2. NAČTENÍ MODELU A LoRA KONFIGURACE
# ==============================================================================
print("\n2. Načítám model 'unsloth/gemma-3-4b-it' a tokenizér...")

# Načtení modelu
model, tokenizer = FastLanguageModel.from_pretrained(
    # Používáme tebou vybraný optimalizovaný model
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Aplikace LoRA adaptérů
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,             # Rank matice (r)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,    # Škálovací faktor
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Optimalizace VRAM
    random_state = 3407,
)

print("Model připraven pro trénink s QLoRA.")


# ==============================================================================
# 3. PŘÍPRAVA VLASTNÍCH DAT (Nahrazení Alpaca datasetu)
# ==============================================================================
print("\n3. Načítám a formátuji TVÉ datové sady...")

# Kontrola souborů a načtení
if not (os.path.exists(TRAIN_FILE) and os.path.exists(TEST_FILE)):
    print(f"❌ KRITICKÁ CHYBA: Soubory {TRAIN_FILE} a {TEST_FILE} nebyly nalezeny.")
    raise FileNotFoundError("Chybí vstupní JSONL soubory. Nahraj je do Colab adresáře a spusť znovu.")

datasets = load_dataset("json", data_files={"train": TRAIN_FILE, "test": TEST_FILE})

# Definice formátu, na kterém model trénoval (Alpaca-like)
# POZOR: POUŽÍVÁME ZDE TŘI ZNAKY ### JAKO V UNLSOTH ŠABLONĚ
ALPACA_PROMPT = """### Instruction:{}### Input:{}### Response:{}"""
EOS_TOKEN = tokenizer.eos_token # Token pro konec sekvence

def formatting_prompts_func(examples):
    texts = []
    # Zde procházíme sloupce z tvého JSONL souboru
    for instruction, input_text, output_text in zip(examples["instruction"], examples["input"], examples["output"]):
        # Musíme přidat EOS_TOKEN, jinak model generuje donekonečna!
        text = ALPACA_PROMPT.format(instruction, input_text, output_text) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Aplikace formátování
datasets = datasets.map(formatting_prompts_func, batched = True,)

print(f"Data načtena a naformátována. Train: {datasets['train'].num_rows} ks, Test: {datasets['test'].num_rows} ks.")





1. Instaluji knihovny...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Importy hotovy. Pokračuji načtením modelu.

2. Načítám model 'unsloth/gemma-3-4b-it' a tokenizér...
==((====))==  Unsloth 2025.11.2: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients
Model připraven pro trénink s QLoRA.

3. Načítám a formátu

trenink (riziko overfittingu ale zdalo se jako nutnost pri menší teprature a malemu mnotsvi finetunovacích dat) bere se však best state model tedy vyberes dohormady njelepší stav modelu po všech epochách

In [2]:
# ==============================================================================
# 4. TRÉNOVÁNÍ MODELU
# ==============================================================================


if datasets["train"].num_rows > 0:
    # Zde definujeme, že se použije SFTTrainer
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = datasets["train"],
        eval_dataset = datasets["test"],
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            num_train_epochs = 10,  # <<< ZVÝŠENO NA 10 EPOCH
            learning_rate = 2e-5,   # <<< SNÍŽENO pro stabilnější konvergenci
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 1,
            optim = "adamw_8bit",
            weight_decay = 0.05,    # <<< ZVÝŠENO pro boj s overfittingem
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = OUTPUT_DIR,
            eval_strategy = "epoch",
            save_strategy = "epoch",
            load_best_model_at_end = True,
            report_to = "none",
        ),
    )

    print("\n--- Spouštím Fine-tuning (10 Epoch, Nové parametry) ---")
    # Tento trénink potrvá zhruba 25-35 minut na T4 GPU
    trainer_stats = trainer.train()

    # Uložení finálního modelu (vynucené 4bit uložení, protože je to finální krok)
    model.save_pretrained_merged(OUTPUT_DIR + "_merged_10ep", tokenizer, save_method = "merged_4bit_forced")

    print(f"\n✅ Trénink dokončen! Finální model uložen jako '{OUTPUT_DIR}_merged_10ep'.")


Unsloth: Switching to float32 training since model cannot work with float16

--- Spouštím Fine-tuning (10 Epoch, Nové parametry) ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 10 | Total steps = 130
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,788,480 of 4,332,867,952 (0.76% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,2.033500,3.474569
2,1.599000,3.087332
3,1.449900,2.878071
4,1.441600,2.708075
5,1.327900,2.601478
6,0.944100,2.555156
7,1.164000,2.533703
8,1.217800,2.521096
9,1.034000,2.515813
10,0.956300,2.514202


Unsloth: Not an error, but Gemma3ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Unsloth: Merging LoRA weights into 4bit model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Unsloth: Merging finished.
Unsloth: Found skipped modules: ['model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj', 'model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj', 'model.vision_tower.vision_model.encoder.layers.0.self_attn.q_proj', 'model.vision_tower.vision_model.encoder.layers.0.self_attn.out_proj', 'model.vision_tower.vision_model.encoder.layers.0.mlp.fc1', 'model.vision_tower.vision_model.encoder.layers.0.mlp.fc2', 'model.vision_tower.vision_model.encoder.layers.1.self_attn.k_proj', 'model.vision_tower.vision_model.encoder.layers.1.self_attn.v_proj', 'model.vision_tower.vision_model.encoder.layers.1.self_attn.q_proj', 'model.vision_tower.vision_model.encoder.layers.1.self_attn.out_proj', 'model.vision_tower.vision_model.encoder.layers.1.mlp.fc1', 'model.vision_tower.vision_model.encoder.layers.1.mlp.fc2', 'model.vision_tower.vision_model.encoder.layers.2.self_attn.k_proj', 'model.vision_tower.vision_model.encoder.layers.2.self_attn.v_proj', 'model

inference a test
(zde byl problém že model stejně generoval  výstup v anglicitne ze finetunig na tak malých datech nepomohl model přeučit na odpovedi v čestine, vynutili sme si tak at odpovida v cestine
) je potřeba dávat pozor na temprature!!


In [3]:
# ==============================================================================
# 1. POKUS: AGRESIVNÍ TEST (BEZ INSTRUKCE)
# ==============================================================================
print("\n--- 1. POKUS: AGRESIVNÍ TEST ---")

# Zopakujeme definice
ALPACA_PROMPT_AGRESSIVE = """### Input:{}### Response:{}""" # Vynecháváme Instruction:
FastLanguageModel.for_inference(model)

vlastni_interpret = "Zdeněk Svěrák a Jaroslav Uhlíř"
vlastni_nazev = "Statistika"
vlastni_text = """
Je statisticky dokázáno,
že slunce vyjde každé ráno,
a i když je tma jako v ranci,
noc nemá celkem žádnou šanci.
Statistika nuda je,
má však cenné údaje,
neklesejme na mysli,
ona nám to vyčíslí.
Když drak si z nosu síru pouští
a Honza na něj číhá v houští,
pak statistika předpovídá,
že nestvůra už neposnídá.
"""

# --- Připravíme Input a Hint pro češtinu ---
input_data = f"Interpret: {vlastni_interpret}\nNázev: {vlastni_nazev}\nText:\n{vlastni_text}"
CZECH_FORMAT_HINT = "Alt. název: "

# Nový prompt: Pouze INPUT + začátek RESPONSE
prompt_input = ALPACA_PROMPT_AGRESSIVE.format(input_data, CZECH_FORMAT_HINT)

# --- Generování (s T=0.3) ---
inputs = tokenizer(text=[prompt_input], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature = 0.3, # Nízká teplota pro stabilitu
    eos_token_id = tokenizer.eos_token_id,
    pad_token_id = tokenizer.pad_token_id,
)
final_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- VSTUP K ANALÝZE (AGRESIVNÍ) ---")
print(prompt_input)

print("--- VÝSLEDEK ---")
if CZECH_FORMAT_HINT in final_output:
    response_part = final_output.split(CZECH_FORMAT_HINT)[1].split(tokenizer.eos_token)[0].strip()
    print(CZECH_FORMAT_HINT + response_part)
else:
    print("Generování selhalo. Celý výstup:")
    print(final_output)


--- 1. POKUS: AGRESIVNÍ TEST ---

--- VSTUP K ANALÝZE (AGRESIVNÍ) ---
### Input:Interpret: Zdeněk Svěrák a Jaroslav Uhlíř
Název: Statistika
Text:

Je statisticky dokázáno,
že slunce vyjde každé ráno,
a i když je tma jako v ranci,
noc nemá celkem žádnou šanci.
Statistika nuda je,
má však cenné údaje,
neklesejme na mysli,
ona nám to vyčíslí.
Když drak si z nosu síru pouští
a Honza na něj číhá v houští,
pak statistika předpovídá,
že nestvůra už neposnídá.
### Response:Alt. název: 
--- VÝSLEDEK ---
Alt. název: Statistická jistota


In [4]:
# ==============================================================================
# 2. POKUS: MALÁ INSTRUKCE
# ==============================================================================
print("\n--- 2. POKUS: MALÁ INSTRUKCE ---")

# Použijeme původní, plný ALPACA_PROMPT
ALPACA_PROMPT = """### Instruction:{}### Input:{}### Response:{}"""

# --- Instrukce a Hint ---
# Krátká instrukce (model by měl vědět, že to má být česky)
instruction_short = "Analyzuj píseň, vytvoř alternativní název, shrnutí (1 větu) a ohodnoť náladu (1-5)."

vlastni_interpret = "Zdeněk Svěrák a Jaroslav Uhlíř"
vlastni_nazev = "Statistika"
vlastni_text = """
Je statisticky dokázáno,
že slunce vyjde každé ráno,
a i když je tma jako v ranci,
noc nemá celkem žádnou šanci.
Statistika nuda je,
má však cenné údaje,
neklesejme na mysli,
ona nám to vyčíslí.
Když drak si z nosu síru pouští
a Honza na něj číhá v houští,
pak statistika předpovídá,
že nestvůra už neposnídá.
"""

input_data = f"Interpret: {vlastni_interpret}\nNázev: {vlastni_nazev}\nText:\n{vlastni_text}"
CZECH_FORMAT_HINT = "Alt. název: "

# Vložíme krátkou instrukci, Input a Hint do Response
prompt_input = ALPACA_PROMPT.format(instruction_short, input_data, CZECH_FORMAT_HINT)

# --- Generování (s T=0.3) ---
inputs = tokenizer(text=[prompt_input], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature = 0.3,
    eos_token_id = tokenizer.eos_token_id,
    pad_token_id = tokenizer.pad_token_id,
)
final_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- VSTUP K ANALÝZE (MALÁ INSTRUKCE) ---")
print(prompt_input)

print("--- VÝSLEDEK ---")
if CZECH_FORMAT_HINT in final_output:
    response_part = final_output.split(CZECH_FORMAT_HINT)[1].split(tokenizer.eos_token)[0].strip()
    print(CZECH_FORMAT_HINT + response_part)
else:
    print("Generování selhalo. Celý výstup:")
    print(final_output)


--- 2. POKUS: MALÁ INSTRUKCE ---

--- VSTUP K ANALÝZE (MALÁ INSTRUKCE) ---
### Instruction:Analyzuj píseň, vytvoř alternativní název, shrnutí (1 větu) a ohodnoť náladu (1-5).### Input:Interpret: Zdeněk Svěrák a Jaroslav Uhlíř
Název: Statistika
Text:

Je statisticky dokázáno,
že slunce vyjde každé ráno,
a i když je tma jako v ranci,
noc nemá celkem žádnou šanci.
Statistika nuda je,
má však cenné údaje,
neklesejme na mysli,
ona nám to vyčíslí.
Když drak si z nosu síru pouští
a Honza na něj číhá v houští,
pak statistika předpovídá,
že nestvůra už neposnídá.
### Response:Alt. název: 
--- VÝSLEDEK ---
Alt. název: 24 hodin
Shrnutí: Píseň popisuje nezměnlivost světa a důležitost statistiky pro pochopení reality.
Nálada: 3


In [5]:
# ==============================================================================
# 6. SROVNÁVACÍ INFERENCE SE ZÁKLADNÍM MODELEM
# ==============================================================================
print("\n--- 6. SROVNÁNÍ: Načítám základní model Gemma-3-4B-it ---")

# Konfigurace základního modelu (stejné hyperparametry jako na začátku)
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Načtení modelu BEZ LoRA adaptérů!
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Přepnutí základního modelu do režimu inference
FastLanguageModel.for_inference(base_model)

# --- Nastavení vstupu (Použijeme stejný prompt jako v Kroku 5) ---
vlastni_interpret = "Zdeněk Svěrák a Jaroslav Uhlíř"
vlastni_nazev = "Statistika"
vlastni_text = """
Je statisticky dokázáno,
že slunce vyjde každé ráno,
a i když je tma jako v ranci,
noc nemá celkem žádnou šanci.
Statistika nuda je,
má však cenné údaje,
neklesejme na mysli,
ona nám to vyčíslí.
Když drak si z nosu síru pouští
a Honza na něj číhá v houští,
pak statistika předpovídá,
že nestvůra už neposnídá.
"""

# Krátká instrukce a Hint pro češtinu
ALPACA_PROMPT = """### Instruction:{}### Input:{}### Response:{}"""
instruction_short = "Analyzuj píseň, vytvoř alternativní název, shrnutí (1 větu) a ohodnoť náladu (1-5). Odpověz POUZE v češtině a použij oddělovač '|'."
input_data = f"Interpret: {vlastni_interpret}\nNázev: {vlastni_nazev}\nText:\n{vlastni_text}"
CZECH_FORMAT_HINT = "Alt. název: "

prompt_input = ALPACA_PROMPT.format(instruction_short, input_data, CZECH_FORMAT_HINT)

# --- Generování základním modelem ---
inputs = base_tokenizer(text=[prompt_input], return_tensors = "pt").to("cuda")

outputs = base_model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature = 0.3,
    eos_token_id = base_tokenizer.eos_token_id,
    pad_token_id = base_tokenizer.pad_token_id,
)
base_output = base_tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- Zobrazení Výsledků ---
print("\n-----------------------------------------------------")
print("🔥 VÝSLEDEK ZÁKLADNÍHO (NETRÉNOVANÉHO) MODELU:")
print("-----------------------------------------------------")
print("Očekávané chování: Anglická odpověď nebo nesprávný formát.")

if CZECH_FORMAT_HINT in base_output:
    response_part = base_output.split(CZECH_FORMAT_HINT)[1].split(base_tokenizer.eos_token)[0].strip()
    print(CZECH_FORMAT_HINT + response_part)
else:
    print("Celý výstup: ")
    print(base_output)

print("\n\n-----------------------------------------------------")
print("✅ VÝSLEDEK TVÉHO FINE-TUNED MODELU (Z Kroku 5):")
print("-----------------------------------------------------")
# Pro srovnání zopakujeme výstup tvého trénovaného modelu (proměnná 'final_output' z Kroku 5)
print(final_output)
# Poznámka: Tato proměnná 'final_output' musí být definována v předchozí buňce, proto buď opatrný při spouštění.


--- 6. SROVNÁNÍ: Načítám základní model Gemma-3-4B-it ---
==((====))==  Unsloth 2025.11.2: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.

-----------------------------------------------------
🔥 VÝSLEDEK ZÁKLADNÍHO (NETRÉNOVANÉHO) MODELU:
-----------------------------------------------------
Očekávané chování: Anglická odpověď nebo nesprávný formát.
Alt. název: 24 hodin
Shrnutí: Píseň mluví o jistotě a předvídatelnosti světa, i když se zdá být chaotický.
Nálada: 3|


-------------